In [1]:
"""
使用训练好的DenseNet161模型对新数据进行推理并为恶性预测添加置信度标记
- 预测判断：torch.max（等价于固定阈值0.5），二分类标准做法
- 文件名置信度：保留4位小数原始数值
- CONFIDENCE_THRESHOLD：只用于筛选哪些恶性图片值得复制
"""

import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import shutil

from shilab_classifier.models.model_definitions import create_model
from shilab_classifier.training.preprocessing import pad_to_square_transform

# ============================================================
# 配置参数
# ============================================================
MODEL_NAME   = "DenseNet161"
MODEL_PATH   = r"H:\limr\project\urine\result\classification_model\UR_SingleCell_seed300\train_val_models\DenseNet161\densenet161_fold_4.pth"
NEW_DATA_DIR = r"H:\limr\project\urine\rawdata\classification_model\UR_SingleCell\malignant"
OUTPUT_DIR   = r"H:\limr\project\urine\result\binary_infer_codetest\output"
MALIGNANT_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'malignant_images')

BATCH_SIZE  = 16
NUM_WORKERS = 0
NUM_CLASSES = 2
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# ⚠️ 必须和训练时完全一致！
MEAN = [0.5992, 0.5755, 0.7237]
STD  = [0.2433, 0.2332, 0.1263]

# ⚠️ 必须和训练时完全一致！
IDX_TO_CLASS = {0: "benign", 1: "malignant"}

# 恶性图片复制的置信度筛选阈值
# 只影响"哪些恶性图片被复制到输出文件夹"，不影响预测判断本身
CONFIDENCE_THRESHOLD = 0.5


# ============================================================
# 数据加载类
# ============================================================
class UnlabeledImageDataset(Dataset):
    """处理单文件夹结构的无标签图像数据集"""

    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform   = transform
        exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
        self.image_files = sorted([
            f for f in os.listdir(folder_path)
            if f.lower().endswith(exts)
        ])
        self.samples = [(os.path.join(folder_path, f), 0) for f in self.image_files]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.samples[idx][0]
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, 0


# ============================================================
# 模型权重加载（兼容多种保存格式）
# ============================================================
def load_model_weights(model, model_path, device):
    checkpoint = torch.load(model_path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print("已加载 checkpoint['model_state_dict']")
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
        print("已加载 checkpoint['state_dict']")
    else:
        model.load_state_dict(checkpoint)
        print("已加载直接保存的 state_dict")
    return model


# ============================================================
# 主函数
# ============================================================
def predict_new_data():
    """对新收集的数据进行预测并为恶性预测添加置信度标记"""
    print(f"\n{'='*60}")
    print(f"使用 {MODEL_NAME} 模型预测新数据")
    print(f"{'='*60}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(MALIGNANT_OUTPUT_DIR, exist_ok=True)

    # ----------------------------------------------------------
    # 1. 加载模型
    # ----------------------------------------------------------
    print(f"加载模型权重: {MODEL_PATH}")
    model = create_model(MODEL_NAME, num_classes=NUM_CLASSES)
    model = load_model_weights(model, MODEL_PATH, DEVICE)
    model = model.to(DEVICE)
    model.eval()

    # ----------------------------------------------------------
    # 2. 数据预处理（必须和训练时完全一致）
    # ----------------------------------------------------------
    print(f"使用标准化参数 - 均值: {MEAN}, 标准差: {STD}")
    transform = transforms.Compose([
        transforms.Lambda(pad_to_square_transform),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ])

    # ----------------------------------------------------------
    # 3. 加载数据
    # ----------------------------------------------------------
    print(f"加载新数据: {NEW_DATA_DIR}")

    # 过滤隐藏文件夹（如 .ipynb_checkpoints）再判断目录结构
    has_subfolders = any(
        os.path.isdir(os.path.join(NEW_DATA_DIR, d))
        for d in os.listdir(NEW_DATA_DIR)
        if not d.startswith('.')
    )

    if has_subfolders:
        print("检测到子文件夹结构，使用ImageFolder加载数据...")
        print("⚠️  注意：请确认下方打印的类别映射和训练时一致！")
        try:
            dataset    = datasets.ImageFolder(NEW_DATA_DIR, transform=transform)
            dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                                    shuffle=False, num_workers=NUM_WORKERS)
            print(f"找到 {len(dataset)} 张图片，分为 {len(dataset.class_to_idx)} 个类别")
            for class_name, idx in dataset.class_to_idx.items():
                print(f"  - 文件夹类别 {idx}: {class_name}")
        except Exception as e:
            print(f"加载数据时出错: {e}")
            return
    else:
        print("检测到单一文件夹结构，使用自定义数据集加载...")
        dataset    = UnlabeledImageDataset(NEW_DATA_DIR, transform=transform)
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS)
        print(f"找到 {len(dataset)} 张图片")

    if len(dataset) == 0:
        print("错误：没有找到任何图片，请检查路径和文件格式。")
        return

    # ----------------------------------------------------------
    # 4. 推理
    # ----------------------------------------------------------
    print("\n开始预测...")
    all_paths  = []
    all_logits = []
    all_probs  = []
    all_preds  = []

    with torch.no_grad():
        for batch_idx, (inputs, _) in enumerate(dataloader):
            if batch_idx % 10 == 0:
                print(f"  预测批次 {batch_idx + 1}/{len(dataloader)}")

            # 获取当前批次图像路径
            batch_size_actual = inputs.size(0)
            start_idx   = batch_idx * BATCH_SIZE
            batch_paths = [
                dataset.samples[i][0]
                for i in range(start_idx, min(start_idx + batch_size_actual, len(dataset)))
            ]

            inputs  = inputs.to(DEVICE)
            outputs = model(inputs)

            # 兼容 Inception 等多输出模型
            if isinstance(outputs, tuple):
                outputs = outputs[0]

            probs = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)

            all_paths.extend(batch_paths)
            all_logits.append(outputs.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_preds.append(predicted.cpu().numpy())

    # ----------------------------------------------------------
    # 5. 整理结果
    # ----------------------------------------------------------
    all_logits = np.concatenate(all_logits, axis=0)
    all_probs  = np.concatenate(all_probs,  axis=0)
    all_preds  = np.concatenate(all_preds,  axis=0)

    results_df = pd.DataFrame({
        'image_path'     : all_paths,
        'filename'       : [os.path.basename(p) for p in all_paths],
        'predicted_idx'  : all_preds,
        'predicted_class': [IDX_TO_CLASS[int(idx)] for idx in all_preds],
        'logit_benign'   : all_logits[:, 0],
        'logit_malignant': all_logits[:, 1],
        'prob_benign'    : all_probs[:, 0],
        'prob_malignant' : all_probs[:, 1],
    })

    results_path = os.path.join(OUTPUT_DIR, 'prediction_results.csv')
    results_df.to_csv(results_path, index=False, encoding='utf-8-sig')
    print(f"\n预测结果已保存到 {results_path}")

    # 统计
    class_counts = results_df['predicted_class'].value_counts()
    print("\n预测类别统计:")
    for class_name, count in class_counts.items():
        print(f"  - {class_name}: {count} 张 ({count / len(results_df) * 100:.1f}%)")

    # ----------------------------------------------------------
    # 6. 处理恶性图片
    # ----------------------------------------------------------
    print("\n开始处理恶性预测图片...")
    process_malignant_predictions(results_df)

    # ----------------------------------------------------------
    # 7. 绘制概率分布图
    # ----------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.hist(results_df['prob_malignant'], bins=20, alpha=0.7,
             color='steelblue', edgecolor='white')
    plt.axvline(x=CONFIDENCE_THRESHOLD, color='red', linestyle='--',
                label=f'筛选阈值 = {CONFIDENCE_THRESHOLD}')
    plt.title('恶性预测概率分布')
    plt.xlabel('恶性概率')
    plt.ylabel('图片数量')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    prob_hist_path = os.path.join(OUTPUT_DIR, 'malignant_probability_distribution.png')
    plt.savefig(prob_hist_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"预测概率分布图已保存到 {prob_hist_path}")

    print("\n✅ 预测完成！")
    return results_df


# ============================================================
# 处理恶性预测图片
# ============================================================
def process_malignant_predictions(results_df):
    """
    复制恶性预测图片到输出目录，文件名附带4位小数置信度
    筛选条件：predicted_class == 'malignant' 且 prob_malignant >= CONFIDENCE_THRESHOLD
    """
    malignant_predictions = results_df[
        (results_df['predicted_class'] == 'malignant') &
        (results_df['prob_malignant'] >= CONFIDENCE_THRESHOLD)
    ]

    print(f"找到 {len(malignant_predictions)} 张恶性预测图片（置信度 ≥ {CONFIDENCE_THRESHOLD}）")

    if len(malignant_predictions) == 0:
        print("没有找到符合条件的恶性预测图片")
        return

    processed_count    = 0
    renamed_files_info = []

    for _, row in malignant_predictions.iterrows():
        original_path = row['image_path']
        confidence    = row['prob_malignant']

        name, ext = os.path.splitext(os.path.basename(original_path))

        # ✅ 保留4位小数原始数值，格式统一且可读
        # 示例：0.9521 → "cell_001_malignant_0.9521.jpg"
        #       1.0000 → "cell_002_malignant_1.0000.jpg"
        confidence_str = f"{confidence:.4f}"
        new_filename   = f"{name}_malignant_{confidence_str}{ext}"
        new_path       = os.path.join(MALIGNANT_OUTPUT_DIR, new_filename)

        try:
            shutil.copy2(original_path, new_path)
            processed_count += 1
            renamed_files_info.append({
                'original_path': original_path,
                'new_path'     : new_path,
                'confidence'   : confidence,
            })
        except Exception as e:
            print(f"  ⚠️  处理文件 {original_path} 时出错: {e}")

    if renamed_files_info:
        renamed_df   = pd.DataFrame(renamed_files_info)
        renamed_path = os.path.join(OUTPUT_DIR, 'malignant_images_with_confidence.csv')
        renamed_df.to_csv(renamed_path, index=False, encoding='utf-8-sig')
        print(f"已处理 {processed_count} 张恶性图片，详情保存到 {renamed_path}")
        print(f"带置信度的恶性图片已保存到 {MALIGNANT_OUTPUT_DIR}")


# ============================================================
# 主程序入口
# ============================================================
if __name__ == "__main__":
    predict_new_data()


使用 DenseNet161 模型预测新数据
加载模型权重: H:\limr\project\urine\result\classification_model\UR_SingleCell_seed300\train_val_models\DenseNet161\densenet161_fold_4.pth


c:\Users\ROG\.conda\envs\learn\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ROG\.conda\envs\learn\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet161_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet161_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


已加载直接保存的 state_dict
使用标准化参数 - 均值: [0.5992, 0.5755, 0.7237], 标准差: [0.2433, 0.2332, 0.1263]
加载新数据: H:\limr\project\urine\rawdata\classification_model\UR_SingleCell\malignant
检测到单一文件夹结构，使用自定义数据集加载...
找到 679 张图片

开始预测...
  预测批次 1/43
  预测批次 11/43
  预测批次 21/43
  预测批次 31/43
  预测批次 41/43

预测结果已保存到 H:\limr\project\urine\result\binary_infer_codetest\output\prediction_results.csv

预测类别统计:
  - malignant: 676 张 (99.6%)
  - benign: 3 张 (0.4%)

开始处理恶性预测图片...
找到 676 张恶性预测图片（置信度 ≥ 0.5）
已处理 676 张恶性图片，详情保存到 H:\limr\project\urine\result\binary_infer_codetest\output\malignant_images_with_confidence.csv
带置信度的恶性图片已保存到 H:\limr\project\urine\result\binary_infer_codetest\output\malignant_images


C:\Users\ROG\AppData\Local\Temp\ipykernel_38732\2924175902.py:242: UserWarning: Glyph 24694 (\N{CJK UNIFIED IDEOGRAPH-6076}) missing from current font.
  plt.tight_layout()
C:\Users\ROG\AppData\Local\Temp\ipykernel_38732\2924175902.py:242: UserWarning: Glyph 24615 (\N{CJK UNIFIED IDEOGRAPH-6027}) missing from current font.
  plt.tight_layout()
C:\Users\ROG\AppData\Local\Temp\ipykernel_38732\2924175902.py:242: UserWarning: Glyph 27010 (\N{CJK UNIFIED IDEOGRAPH-6982}) missing from current font.
  plt.tight_layout()
C:\Users\ROG\AppData\Local\Temp\ipykernel_38732\2924175902.py:242: UserWarning: Glyph 29575 (\N{CJK UNIFIED IDEOGRAPH-7387}) missing from current font.
  plt.tight_layout()
C:\Users\ROG\AppData\Local\Temp\ipykernel_38732\2924175902.py:242: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from current font.
  plt.tight_layout()
C:\Users\ROG\AppData\Local\Temp\ipykernel_38732\2924175902.py:242: UserWarning: Glyph 29255 (\N{CJK UNIFIED IDEOGRAPH-7247}) missing fr

预测概率分布图已保存到 H:\limr\project\urine\result\binary_infer_codetest\output\malignant_probability_distribution.png

✅ 预测完成！
